# 02 · Features, backtest and baseline calibration

**Purpose.** Measure whether graph-derived features carry real signal, especially where the bank score is confused, and write the first
**likelihood-ratio table** the backend's judge can use.

**Method, in one paragraph.** For every closed case we take its first suspicious transaction and compute features **as of that transaction's
time** (nothing later). We split by time: train on cases opened before 1 October, test on October. Nothing from the test month is used to fit anything.

**Honesty note.** Closed history is 84% fraud, and its *innocent* cases were all score-triggered (score ≥ 0.8). Numbers here show that signal exists.
They are a starting point for calibration, not the final probabilities; see section 7.

## 1 · Setup

In [1]:
import sys
sys.path.insert(0, "../src")            # so `import fraudpy` works from the notebooks folder

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fraudpy.data import get_con, DATA_DIR, ROOT

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.figsize": (8, 3.6), "axes.spines.top": False, "axes.spines.right": False})

con = get_con()                          # slim DuckDB tables, built once and cached in data/work.duckdb
q = lambda sql, *p: con.execute(sql, list(p)).df()   # run SQL, get a DataFrame
print("data folder:", DATA_DIR)

data folder: D:\Coding\hhg-task\Dataset


## 2 · As-of features for every closed case

Each case contributes one row: its first fraud transaction (or the single transaction for a cleared case). The features are the ones the agent's signatures use:

| feature | meaning |
|---|---|
| `new_dev` | identity record marks the device `New` |
| `proxy` | device is behind a proxy (`id_23`) |
| `dev_self0` | this card never used this device profile before |
| `dev_others_30d` | other customers on the same device profile in the last 30 days |
| `reg_new` | billing region never seen on this card |
| `prod_new` | product code never used on this card |
| `over_p95` | amount above the card's own 95th percentile |
| `amt_ratio` | log amount relative to the card's median |
| `burst_2h` | transactions on the card in the previous 2 hours |
| `hub` | card with an implausible volume for one person |

All history is strictly **before** the case's transaction time.

In [2]:
have = con.execute("select count(*) from information_schema.tables where table_name = 'fx'").fetchone()[0]
if not have:
    con.execute("""
    create table fx as
    with f as (
      select cc.case_id, cc.outcome, cc.pattern, cc.opened_at, t.TransactionID tid, t.customer_id, t.ts, t.channel, t.amt, t.prod, t.addr1,
             t.risk_score rs, coalesce(t.card4,'~') c4, coalesce(t.card6,'~') c6
      from (select *, coalesce(first_fraud_txn_id, split_part(txn_ids,'|',1)::bigint) ftid from cc) cc
      join tx t on t.TransactionID = cc.ftid)
    select f.*,
      (select count(*) from tx h where h.customer_id=f.customer_id and coalesce(h.card4,'~')=f.c4 and coalesce(h.card6,'~')=f.c6 and h.ts<f.ts) n_prior,
      (select coalesce(median(h.amt),0) from tx h where h.customer_id=f.customer_id and coalesce(h.card4,'~')=f.c4 and coalesce(h.card6,'~')=f.c6 and h.ts<f.ts) med_prior,
      (select coalesce(quantile_cont(h.amt,0.95),0) from tx h where h.customer_id=f.customer_id and coalesce(h.card4,'~')=f.c4 and coalesce(h.card6,'~')=f.c6 and h.ts<f.ts) p95_prior,
      (select count(*) from tx h where h.customer_id=f.customer_id and coalesce(h.card4,'~')=f.c4 and coalesce(h.card6,'~')=f.c6 and h.ts<f.ts and h.prod=f.prod) n_prod_seen,
      (select count(*) from tx h where h.customer_id=f.customer_id and coalesce(h.card4,'~')=f.c4 and coalesce(h.card6,'~')=f.c6 and h.ts<f.ts and h.addr1=f.addr1) n_reg_seen,
      (select count(*) from tx h where h.customer_id=f.customer_id and coalesce(h.card4,'~')=f.c4 and coalesce(h.card6,'~')=f.c6 and h.ts<f.ts and h.ts>=f.ts-interval 2 hour) n_2h
    from f""")
    con.execute("""
    create table fx2 as
    select fx.*, d.id_15, d.id_23, d.dprof,
      (select count(*) from idn d2 join tx t2 using (TransactionID) where d2.dprof=d.dprof and t2.customer_id=fx.customer_id and t2.ts<fx.ts) dev_prior_self,
      (select count(distinct t2.customer_id) from idn d2 join tx t2 using (TransactionID) where d2.dprof=d.dprof and t2.customer_id<>fx.customer_id and t2.ts<fx.ts and t2.ts>=fx.ts-interval 30 day) dev_others_30d
    from fx left join idn d on d.TransactionID = fx.tid""")
df = q("select * from fx2")
df["y"] = (df.outcome == "confirmed_fraud").astype(int)
df["online"] = (df.channel == "online").astype(int)
df["amt_ratio"] = np.log1p(df.amt) - np.log1p(df.med_prior)
df["over_p95"] = (df.amt > df.p95_prior).astype(int)
df["prod_new"] = (df.n_prod_seen == 0).astype(int)
df["reg_new"] = ((df.n_reg_seen == 0) & df.addr1.notna()).astype(int)
df["new_dev"] = (df.id_15 == "New").astype(int)
df["proxy"] = df.id_23.fillna("").str.contains("PROXY").astype(int)
df["dev_self0"] = (df.dprof.notna() & (df.dev_prior_self == 0)).astype(int)
df["dev_others"] = np.log1p(df.dev_others_30d.fillna(0))
df["hub"] = (df.n_prior > 3000).astype(int)
df["logn"] = np.log1p(df.n_prior)
df["burst_2h"] = np.log1p(df.n_2h)
df["rs"] = df.rs.astype(float)
print(len(df), "closed cases;", int(df.y.sum()), "fraud,", int((1 - df.y).sum()), "cleared")

5565 closed cases; 4665 fraud, 900 cleared


## 3 · Fraud versus false alarm, feature by feature

How different are the two groups on each feature? (Mean of a 0/1 feature is the share of cases where it is true.)

In [3]:
FEATS = ["rs", "online", "amt_ratio", "over_p95", "prod_new", "reg_new", "new_dev", "proxy", "dev_self0", "dev_others", "hub", "logn", "burst_2h"]
df.groupby("outcome")[FEATS].mean().T.round(3)

outcome,cleared,confirmed_fraud
rs,0.881,0.475
online,0.848,0.556
amt_ratio,0.601,0.591
over_p95,0.338,0.207
prod_new,0.207,0.113
reg_new,0.456,0.152
new_dev,0.832,0.181
proxy,0.068,0.030
dev_self0,0.716,0.367
dev_others,2.790,1.703


## 4 · The backtest: score alone, graph alone, both

Train on cases opened **before 1 October 2016**, test on **October**. Metric: ROC AUC (how well the model ranks fraud above false alarm) and Brier score (probability accuracy).
We use gradient boosting only as a *measuring stick* for how much signal there is; the production judge stays a transparent log-odds ledger.

In [4]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss

tr, te = df[df.opened_at < "2016-10-01"], df[df.opened_at >= "2016-10-01"]
print(f"train {len(tr)} cases, test {len(te)} cases, test fraud share {te.y.mean():.3f}")

def fit_eval(cols, data_tr, data_te):
    m = GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=0).fit(data_tr[cols], data_tr.y)
    p = m.predict_proba(data_te[cols])[:, 1]
    return round(roc_auc_score(data_te.y, p), 3), round(brier_score_loss(data_te.y, p), 3)

GRAPH = [f for f in FEATS if f != "rs"]
rows = [("risk score only", *fit_eval(["rs"], tr, te)),
        ("graph features only (no score)", *fit_eval(GRAPH, tr, te)),
        ("score + graph features", *fit_eval(FEATS, tr, te))]
pd.DataFrame(rows, columns=["model", "AUC", "Brier"])

train 4193 cases, test 1372 cases, test fraud share 0.895


,model,AUC,Brier
0,risk score only,0.950,0.054
1,graph features only (no score),0.918,0.068
2,score + graph features,0.989,0.024


### Careful: why "score only" looks good here

Because every cleared case has a score of 0.8 or more while fraud is spread out, the score *appears* to separate the groups in this sample. That is a property of how the sample was
made, not proof that the score is a good detector. The fair test is the next one: inside the zone where **both** kinds of case exist.

In [5]:
zone_tr, zone_te = tr[tr.rs >= 0.8], te[te.rs >= 0.8]
print(f"zone score >= 0.8: train {len(zone_tr)}, test {len(zone_te)}, test fraud share {zone_te.y.mean():.3f}")
pd.DataFrame([
    ("risk score only", *fit_eval(["rs"], zone_tr, zone_te)),
    ("graph features only", *fit_eval(GRAPH, zone_tr, zone_te)),
    ("score + graph features", *fit_eval(FEATS, zone_tr, zone_te)),
], columns=["model (score >= 0.8 only)", "AUC", "Brier"])

zone score >= 0.8: train 1275, test 303, test fraud share 0.525


,model (score >= 0.8 only),AUC,Brier
0,risk score only,0.617,0.246
1,graph features only,0.914,0.113
2,score + graph features,0.918,0.106


**What it means.** Inside the high-score zone (where the bank is confused: roughly half the alerts are false alarms) the score alone is close to useless, and graph
features carry most of the signal. That is the case for a graph-first agent.

## 5 · Do the anonymous Vesta columns carry signal?

The `C`, `D`, `M` and `V` columns have no names, but they are real model features. We test them at the *transaction* level: labelled fraud transactions versus cleared plus a random
sample of unlabelled ones. Evidence based on them must say "engineered features, unnamed", never pretend to know what `V127` means.

In [6]:
DATA = DATA_DIR.as_posix()
cols = [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)] + [f"V{i}" for i in range(1, 340)] + [f"M{i}" for i in range(1, 10)]
sel = ", ".join(f'r."{c}"' for c in cols)
con.execute("create or replace temp table sample_ids as "
            "select tid as TransactionID, 1 y from lab where outcome='confirmed_fraud' union all "
            "select tid, 0 from lab where outcome='cleared' union all "
            "select TransactionID, 0 from (select t.TransactionID from tx t left join lab l on l.tid=t.TransactionID "
            "where l.tid is null and t.risk_score < 0.8 using sample 40000 rows)")
vf = con.execute(f"""select s.y, t.ts, t.risk_score, t.amt, {sel}
    from sample_ids s join tx t using (TransactionID)
    join read_csv('{DATA}/transactions.csv', header=true, sample_size=200000) r on r.TransactionID = s.TransactionID""").df()
for m in [f"M{i}" for i in (1,2,3,5,6,7,8,9)]:
    vf[m] = vf[m].map({"T": 1, "F": 0}).astype(float)
vf["M4"] = vf["M4"].map({"M0": 0, "M1": 1, "M2": 2}).astype(float)
X = [c for c in cols if c in vf.columns]
vtr, vte = vf[vf.ts < "2016-10-01"], vf[vf.ts >= "2016-10-01"]

def auc(cols_):
    m = GradientBoostingClassifier(n_estimators=100, max_depth=3, subsample=0.5, random_state=0).fit(vtr[cols_].fillna(-1), vtr.y)
    return round(roc_auc_score(vte.y, m.predict_proba(vte[cols_].fillna(-1))[:, 1]), 3)

pd.DataFrame([("C/D/M/V only", auc(X)), ("score + amount", auc(["risk_score", "amt"])), ("everything", auc(X + ["risk_score", "amt"]))],
             columns=["features", "AUC (October)"])

,features,AUC (October)
0,C/D/M/V only,0.848
1,score + amount,0.871
2,everything,0.930


**What it means.** The unnamed features add real signal on top of score and amount, so a *numeric lookalike* channel (nearest neighbours in a compressed feature space)
is worth building, as one clearly labelled evidence class.

## 6 · The first likelihood-ratio table

For each binary signal we estimate, **inside the zone where both outcomes exist**:

`LR(present) = P(signal | fraud) / P(signal | cleared)`   and   `LR(absent) = P(no signal | fraud) / P(no signal | cleared)`

Three safeguards, all deliberate:
1. **Smoothing** (+1 pseudo-count) so small samples cannot give infinite ratios.
2. **Shrinkage** (multiply the log-ratio by 0.5) because history is not the exam.
3. **Cap** at ±log(5) so no single signal can dominate.

Because ratios are class-conditional they are *not* affected by the 84% fraud base rate; the **prior** (about 50% for the exam) is applied separately by the judge.

In [7]:
import json, math   # math is used again in section 6b
SIGNALS = {"new_dev": "new_device", "proxy": "proxy", "dev_self0": "device_unseen_by_card", "reg_new": "new_region", "prod_new": "new_product", "over_p95": "amount_over_p95"}
SHRINK, CAP = 0.5, math.log(5)

def lr_table(data):
    fz, cz = data[data.y == 1], data[data.y == 0]
    out = {}
    for col, name in SIGNALS.items():
        pf = (fz[col].sum() + 1) / (len(fz) + 2)
        pc = (cz[col].sum() + 1) / (len(cz) + 2)
        f = lambda x: float(np.clip(SHRINK * x, -CAP, CAP))
        out[name] = {"p_given_fraud": round(pf, 4), "p_given_cleared": round(pc, 4),
                     "log_lr_present": round(f(math.log(pf / pc)), 4), "log_lr_absent": round(f(math.log((1 - pf) / (1 - pc))), 4),
                     "n_fraud": int(len(fz)), "n_cleared": int(len(cz))}
    return out

table = lr_table(zone_tr)                     # fitted on training months only
pd.DataFrame(table).T[["p_given_fraud", "p_given_cleared", "log_lr_present", "log_lr_absent"]]

,p_given_fraud,p_given_cleared,log_lr_present,log_lr_absent
new_device,0.2092,0.8562,-0.7046,0.8523
proxy,0.0537,0.0752,-0.1680,0.0115
device_unseen_by_card,0.3685,0.7533,-0.3575,0.4699
new_region,0.1612,0.4934,-0.5593,0.2521
new_product,0.1056,0.2269,-0.3826,0.0729
amount_over_p95,0.2111,0.3311,-0.2250,0.0825


### Does it generalise? Check on the held-out month

Score every October in-zone case by summing the matching log-ratios (a naive-Bayes ledger, the same idea as the production judge) and measure AUC.

In [8]:
def ledger_score(row):
    s = 0.0
    for col, name in SIGNALS.items():
        s += table[name]["log_lr_present"] if row[col] else table[name]["log_lr_absent"]
    return s

zone_te = zone_te.copy()
zone_te["ledger"] = zone_te.apply(ledger_score, axis=1)
print("held-out AUC of the transparent ledger (score >= 0.8 zone):", round(roc_auc_score(zone_te.y, zone_te.ledger), 3))
print("for comparison, the bank score alone in the same zone:      ", round(roc_auc_score(zone_te.y, zone_te.rs), 3))

held-out AUC of the transparent ledger (score >= 0.8 zone): 0.808
for comparison, the bank score alone in the same zone:       0.409


In [9]:
out = {
    "version": "baseline_v0",
    "note": "Estimated within the score>=0.8 zone (only place both outcomes exist). Fitted on cases opened before 2016-10-01. Shrunk by 0.5, capped at log(5). Prior is applied separately by the judge.",
    "zone": "risk_score >= 0.8",
    "signals": table,
}
path = ROOT / "calibration" / "baseline_v0.json"
path.parent.mkdir(exist_ok=True)
path.write_text(json.dumps(out, indent=2))
print("wrote", path)

wrote D:\Coding\hhg-task\calibration\baseline_v0.json


## 6b · A warning hidden in that table: in this zone, "suspicious-looking" means innocent

Look at the signs. Inside the high-score zone, `new_device`, `new_region` and `device_unseen_by_card` all have **negative** log-ratios: they are *more* common among the
false alarms than among the fraud. That is the opposite of what a naive fraud rule assumes. Two explanations to rule out: (a) a mix effect (most fraud in this zone might be
in-person, where devices do not exist) and (b) something real about how false alarms look. We stratify by channel to separate them.

In [10]:
z = df[df.rs >= 0.8]
print("who is in the zone:")
display(pd.crosstab(z.channel, z.outcome))
rows = []
for ch in ["online", "in_person"]:
    s = z[z.channel == ch]
    f, c = s[s.y == 1], s[s.y == 0]
    for col in ["new_dev", "reg_new", "prod_new", "over_p95"]:
        pf, pc = (f[col].sum() + 1) / (len(f) + 2), (c[col].sum() + 1) / (len(c) + 2)
        rows.append({"channel": ch, "signal": col, "P(signal | fraud)": round(pf, 3), "P(signal | cleared)": round(pc, 3),
                     "raw log-LR when present": round(math.log(pf / pc), 2), "fraud n": len(f), "cleared n": len(c)})
pd.DataFrame(rows)

who is in the zone:


outcome,cleared,confirmed_fraud
channel,,
in_person,137,304
online,763,374


,channel,signal,P(signal | fraud),P(signal | cleared),raw log-LR when present,fraud n,cleared n
0,online,new_dev,0.364,0.980,-0.99,374,763
1,online,reg_new,0.154,0.505,-1.19,374,763
2,online,prod_new,0.128,0.242,-0.64,374,763
3,online,over_p95,0.197,0.226,-0.14,374,763
4,in_person,new_dev,0.003,0.007,-0.79,304,137
5,in_person,reg_new,0.131,0.187,-0.36,304,137
6,in_person,prod_new,0.056,0.022,0.95,304,137
7,in_person,over_p95,0.190,0.957,-1.62,304,137


**What it means.** The sign does *not* flip when we split by channel. Among **online** alerts scoring 0.8 or higher, 98% of the cleared cases show a `New` device against 36% of the fraud.
In this zone the false alarms are the cases that *look* the most suspicious on naive signals (a new phone, a new region, an unusual amount), and the confirmed fraud is comparatively ordinary.
This matches the README: "above 0.7, most flagged transactions turn out to be legitimate".

**Consequence for the agent.** A signal's meaning depends on the bank score around it. Below 0.8, a new device is evidence of fraud (74% of confirmed new-device fraud); at 0.8 and above, it is
evidence of a new phone. So the backend has two signatures: `new_device` (score under 0.8, prosecution, positive) and `new_device_at_high_score` (0.8 and above, defence, negative, equal to the
`new_device` value in `baseline_v0.json`). Additive evidence classes cannot express that interaction on their own, which is why the score zone is used to *select* the signature.

*Caveat.* Below 0.8 there are no cleared cases in history, so we cannot measure the innocent side there; the positive weight outside the zone is a reasoned default, to be checked by the backtest.

## 7 · Limits, and what comes next

- **Population.** Innocent history is only score-triggered alerts, so these ratios say nothing about customer-report cases. Those rely on rule-based defence tests.
- **Selection.** Only investigated transactions have labels; un-cased transactions are not proven innocent (the ring proves that).
- **Prior shift.** The judge must reset the prior to the exam's mix (about half legitimate) and never train on raw class frequency.
- **No tuning on the exam.** The 20 exam cases were never used here.

**Next step in the backend:** the judge loads `calibration/baseline_v0.json` where signal names match, and otherwise falls back to hand-set defaults. The full backtest
(replaying closed cases through the whole agent) replaces this baseline once the agent runs end to end.